# Quarto

A refresher on **Quarto** — an open-source, language-agnostic **scientific publishing system**. You write a plain-text document that mixes prose (Markdown) with executable code cells, and Quarto runs the code, captures the output, and renders the whole thing to HTML, PDF, MS Word, a website, a book, a blog, slides, or a journal article — from one source.

**Domain:** Data Analysis & Research  ·  **runnable:** no — Quarto is a **CLI/publishing tool**, not a Python library. Everything below is shell commands, `.qmd`/`_quarto.yml` snippets, and config, not executed notebook cells. (No fake `print()` output here — you drive Quarto from a terminal, not from inside this kernel.)

## 1. What & Why

**What it is.** Quarto is the successor to **R Markdown**, rebuilt by Posit (formerly RStudio) to be **language-agnostic**. A `.qmd` file is Pandoc Markdown plus fenced **code chunks**. When you `quarto render`, it executes the chunks (via **Jupyter** for Python/Julia or **knitr** for R), weaves the results back into the document, and hands the assembled Markdown to **Pandoc**, which produces the final format. One source → many outputs.

**The problem it solves.** Keeping a report and the code that produced its figures in sync. Copy-pasting a matplotlib chart into Word breaks the moment the data changes. With Quarto the figures, tables, and numbers are *regenerated from code every render*, so the document is always consistent with the analysis — **reproducible research** by construction. It also unifies a fragmented toolchain (R Markdown, Jupyter Book, bookdown, Distill, `nbconvert`) under one CLI and one config format.

**When to reach for it.**
- Reproducible analysis reports, papers, and theses where prose and results must stay coupled.
- Documentation sites, technical blogs, and books (multi-page, cross-referenced, searchable).
- Reveal.js / PowerPoint slide decks generated from the same code as your report.
- Polyglot teams — a Python user and an R user can both contribute chunks to one document.
- Turning existing `.ipynb` notebooks into polished published artifacts without rewriting them.

**When NOT to.** For a quick throwaway plot, just run a script — Quarto's value is in *publishing*. For a fully interactive app with callbacks and state, use **Streamlit / Shiny / Dash**; Quarto output is largely static (though it can embed Observable JS, Shiny, or widgets). If your whole team lives in `.ipynb` and never publishes, the rendering layer is overhead you may not need.

## 2. Mental Model

> **Quarto is a pipeline: your `.qmd` is the recipe, an *engine* cooks the code, and *Pandoc* plates it into whatever format you ordered.**

```
  document.qmd
  ┌──────────────┐
  │ YAML header  │  ← title, format, options (the order ticket)
  │ Markdown     │  ← prose, untouched
  │ ```{python}  │  ← code chunks
  │  code...     │
  │ ```          │
  └──────┬───────┘
         │  (1) EXECUTE via engine
         ▼
   Jupyter (py/julia)  or  knitr (R)   ──▶  runs each chunk, captures stdout/plots/tables
         │
         │  (2) WEAVE results back inline → a plain Markdown (.md) with figures embedded
         ▼
      Pandoc   ──▶  (3) CONVERT to the requested target
         │
         ▼
  html · pdf (via LaTeX) · docx · revealjs · website · book · ...
```

The key insight: **execution and rendering are separate stages.** The engine only cares about your code; Pandoc only cares about Markdown. Quarto orchestrates the handoff. This is why the *same* `.qmd` becomes a webpage or a PDF just by changing `format:` — the code runs once, the output is re-plated. It's literate programming (Knuth) with a modern, multi-format back end.

## 3. Key Concepts

- **`.qmd` file.** The source: a YAML front-matter block (`---` delimited) followed by Markdown and fenced code chunks. A plain `.ipynb` also works as a Quarto source — notebook cell metadata replaces the chunk options.
- **YAML front matter.** Document-level config at the top: `title`, `author`, `format`, `execute`, etc. Keys here set defaults for the whole document.
- **Code chunk.** A fenced block `` ```{python} `` … `` ``` ``. The language in braces picks the behavior. Per-chunk options are set with `#|` comment lines at the top of the chunk (e.g. `#| echo: false`).
- **Engine.** What executes the code: **Jupyter** (Python, Julia, any Jupyter kernel) or **knitr** (R, and R-flavored polyglot). Quarto auto-detects from the chunk languages; override with `engine:`.
- **Format / target.** The output type under `format:` — `html`, `pdf`, `docx`, `revealjs`, `gfm`, etc. PDF requires a **LaTeX** install (`quarto install tinytex`).
- **`_quarto.yml`.** Project-level config file. Marks a directory as a **project** and sets shared defaults + a `type:` (`website`, `book`, `manuscript`) and navigation.
- **Execution options.** `eval` (run the code?), `echo` (show the code?), `output` (show results?), `warning`, `error`, `include` — settable globally under `execute:` or per chunk via `#|`.
- **Freeze / cache.** `execute: freeze: true` re-uses stored results so CI doesn't re-run expensive code; `cache: true` memoizes individual chunks. Freeze results live in `_freeze/`.
- **Cross-references.** Label a figure/table/section (`#fig-foo`, `#tbl-bar`, `#sec-intro`) and reference it with `@fig-foo`; Quarto auto-numbers and links.
- **Shortcodes & divs.** `{{< include file.qmd >}}`, `{{< video … >}}`, callout blocks (`::: {.callout-note}`), and column/layout fenced divs for rich, format-aware content.

## 4. Setup

Quarto is a **standalone CLI** (a single binary), not a `pip` package. Install it once for your machine, then use whatever language stack you already have.

```bash
# macOS (Homebrew) — or download the installer from quarto.org/docs/get-started
brew install --cask quarto

# Linux (.deb shown; .rpm and tarball also available)
sudo dpkg -i quarto-*-linux-amd64.deb

quarto --version          # confirm it's on PATH
quarto check              # diagnose engines: Jupyter, knitr, LaTeX, Chromium
```

Then make sure the engine for your language is available:

```bash
# Python path: Quarto renders via Jupyter, so you need jupyter + your packages
pip install jupyter matplotlib pandas

# R path: Quarto renders via knitr
Rscript -e 'install.packages(c("knitr", "rmarkdown"))'

# PDF output needs LaTeX — Quarto bundles a minimal TinyTeX for you
quarto install tinytex
```

The VS Code, RStudio, and JupyterLab extensions add live preview and a visual editor, but everything works from the bare CLI. Note Quarto is **not** Python-runnable from inside this notebook kernel — it's a command you invoke at the shell.

## 5. Worked Examples

Four examples, all CLI/config (nothing runs in *this* kernel): **(1)** a minimal `.qmd` rendered to HTML and PDF, **(2)** chunk options that control what shows, **(3)** a cross-referenced figure, **(4)** a multi-page website project. Save the snippets to files and run the `quarto` commands shown.

### Example 1 — A minimal document, one source → two formats

Create `report.qmd`. The YAML header lists *both* formats; Quarto runs the Python chunk once and plates the result into each target.

````markdown
---
title: "Quarterly Sales"
author: "A. Analyst"
date: 2026-06-24
format:
  html:
    toc: true          # table of contents in the sidebar
    code-fold: true    # readers can expand the code
  pdf:
    documentclass: article
---

## Summary

Revenue grew steadily this quarter. The chart below is regenerated
on every render, so it can never drift from the data.

```{python}
import matplotlib.pyplot as plt
months = ["Apr", "May", "Jun"]
revenue = [120, 135, 158]
plt.bar(months, revenue)
plt.ylabel("Revenue ($k)")
plt.show()
```
````

Render it:

```bash
quarto render report.qmd                 # builds BOTH html and pdf (every format in the header)
quarto render report.qmd --to html       # just one target
quarto preview report.qmd                # live-reloading browser preview while you edit
```

Output lands at `report.html` / `report.pdf`. Change `revenue` and re-render — the bar chart updates itself; you never touch an image file.

### Example 2 — Controlling what the reader sees with chunk options

Execution options decide whether code, output, and warnings appear. Set document-wide defaults under `execute:`, then override per chunk with `#|` lines.

````markdown
---
title: "Clean Report"
format: html
execute:
  echo: false        # hide source code by default (show only results)
  warning: false     # suppress warnings in the rendered doc
---

```{python}
#| echo: true        # ...but DO show the code for this one chunk
#| label: load-data
import pandas as pd
df = pd.read_csv("sales.csv")
df.head()
```

```{python}
#| eval: false       # show the code but DON'T run it (e.g. a slow/destructive call)
df.to_sql("sales", engine, if_exists="replace")
```
````

The cheat sheet for the common options:

| Option | `true` means | Typical use |
|--------|--------------|-------------|
| `eval` | run the chunk | `false` to display code without executing it |
| `echo` | show the source | `false` for results-only reports |
| `output` | show the result | `false` to run for side effects only |
| `warning` / `error` | surface warnings/errors | `error: true` to keep rendering past a failing chunk |
| `include` | include any of the chunk's output | `false` to run silently and emit nothing |

### Example 3 — A cross-referenced, captioned figure

Give a chunk a `label:` starting with `fig-` and a `fig-cap:`, then reference it anywhere with `@fig-...`. Quarto numbers it and inserts a hyperlink — no manual "Figure 3" bookkeeping.

````markdown
As shown in @fig-trend, revenue is accelerating.

```{python}
#| label: fig-trend
#| fig-cap: "Monthly revenue, Q2 2026."
import matplotlib.pyplot as plt
plt.plot(["Apr", "May", "Jun"], [120, 135, 158], marker="o")
plt.show()
```

See also @tbl-summary and @sec-method.
````

The same scheme works for tables (`#| label: tbl-...`, `#| tbl-cap: ...`) and sections (add `{#sec-method}` after a heading). Callouts add semantic emphasis that renders natively per format:

````markdown
::: {.callout-note}
Data excludes refunds. See the appendix for the full methodology.
:::

::: {.callout-warning}
Figures are preliminary and subject to audit.
:::
````

### Example 4 — A multi-page website project

A directory with a `_quarto.yml` becomes a **project**. Quarto renders every `.qmd` and stitches them into a navigable site. Layout:

```
mysite/
├── _quarto.yml        # project config + navigation
├── index.qmd          # home page
├── about.qmd
└── posts/
    └── analysis.qmd
```

`_quarto.yml`:

```yaml
project:
  type: website
  output-dir: _site

website:
  title: "My Research Site"
  navbar:
    left:
      - href: index.qmd
        text: Home
      - href: about.qmd
        text: About
      - href: posts/analysis.qmd
        text: Analysis

format:
  html:
    theme: cosmo       # any Bootswatch theme; applies site-wide
    css: styles.css
```

Build and ship:

```bash
quarto create project website mysite   # scaffold a fresh project (or write the files yourself)
cd mysite
quarto preview                         # serve the whole site with live reload
quarto render                          # build static site into _site/
quarto publish gh-pages                # one-command deploy to GitHub Pages
```

Swap `type: website` for `type: book` to get chapters, a generated table of contents, and PDF/EPUB output from the same content.

## 6. Gotchas & Pitfalls

- **Wrong/absent kernel.** Quarto's Python engine renders through Jupyter. If `quarto render` can't find your environment's packages, you're on the wrong kernel — set `jupyter: python3` in the YAML, or activate the venv before rendering. `quarto check` tells you what it sees.
- **PDF needs LaTeX.** `format: pdf` fails out of the box until you run `quarto install tinytex`. The error (`pdflatex not found`) is unmistakable; the fix is one command.
- **`.qmd` chunk syntax ≠ Jupyter.** Chunk options use `#|` YAML comment lines *inside* the fence, not cell metadata. `#| echo: false` is correct; a bare `echo: false` line is just code.
- **`eval: false` vs `echo: false` confusion.** `eval: false` = don't *run* it (code shown, no output). `echo: false` = run it but don't *show the code* (output shown). Mixing these up silently changes the document.
- **YAML indentation.** The front matter is strict YAML — two-space nesting, colons need a space after them. A mis-indented `format:` block is the most common "why won't it render" bug.
- **Working directory.** Chunks execute from the document's directory by default; relative paths like `read_csv("data.csv")` resolve there, not from where you launched the terminal. In a project, set `execute-dir: project` for consistency.
- **Expensive code re-runs every render.** Without `freeze`/`cache`, every `quarto render` re-executes all chunks — painful in CI. Set `execute: freeze: auto` so unchanged docs reuse stored results from `_freeze/` (commit that folder).
- **`quarto render` builds *every* format in the header.** If you only want HTML during iteration, pass `--to html` instead of waiting on the PDF/LaTeX pass too.
- **It's not interactive by default.** Output is static HTML/PDF. For live widgets you need Observable JS, embedded `ipywidgets` (HTML only), or a Shiny runtime — don't expect Streamlit-style reactivity for free.

## 7. When to Use vs Alternatives

| Tool | Best at | Weakness vs Quarto |
|------|---------|--------------------|
| **Quarto** | Multi-format, multi-language reproducible publishing from one source (reports, sites, books, slides) | Static output; another binary to install; overkill for throwaway analysis |
| **Jupyter / nbconvert** | Interactive exploration; the de-facto notebook format | Weaker publishing — limited cross-refs, theming, multi-page sites, native PDF |
| **R Markdown / bookdown** | The R-world predecessor; deep knitr integration | R-centric, fragmented packages; Quarto is its official, polyglot successor |
| **Jupyter Book** | Python book/website from notebooks | Python/Sphinx-centric; Quarto is more language-agnostic and simpler config |
| **Sphinx / MkDocs** | API/software documentation (docstring extraction, search) | Not built around executable analysis chunks; weaker at "code + narrative + figures" |
| **Streamlit / Shiny / Dash** | Live interactive apps with callbacks and state | Apps, not documents — no static PDF/Word artifact, must run a server |
| **LaTeX (raw)** | Maximum typographic control for print | No code execution; you paste figures in manually; steep authoring |

**Rule of thumb:** producing a *document or site* whose figures must stay in lockstep with code, especially across formats or languages → **Quarto**. Pure interactive exploration → stay in **Jupyter**. A reactive web app → **Streamlit/Shiny**. Pure software API docs → **Sphinx/MkDocs**. Coming from R Markdown → **migrate to Quarto** (it reads most `.Rmd` with minor edits).

## 8. Resources

- **Official site & docs** — install, guide, and the full option reference:
  <https://quarto.org/docs/guide/>
- **Get Started tutorials** (VS Code / RStudio / Jupyter / terminal tracks):
  <https://quarto.org/docs/get-started/>
- **Authoring reference** — Markdown syntax, cross-refs, callouts, figures/tables:
  <https://quarto.org/docs/authoring/markdown-basics.html>
- **Execution options reference** — every `execute:`/`#|` knob, freeze, and caching:
  <https://quarto.org/docs/computations/execution-options.html>
- **Source on GitHub** (issues, releases, the CLI itself):
  <https://github.com/quarto-dev/quarto-cli>
- **Migrating from R Markdown** — what changes and what doesn't:
  <https://quarto.org/docs/faq/rmarkdown.html>

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE